In [4]:
!pip -q install -U ultralytics opencv-python numpy
print('Libraries installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 92.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
Libraries installed.


In [6]:
from pathlib import Path
import urllib.request

program = 'from __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport subprocess\nfrom collections import Counter, defaultdict, deque\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\n\n\n# ============================================================\n# CONFIGURATION\n# ============================================================\n\nSOURCE_VIDEO = Path("/content/intersection.mp4")\nSOURCE_URL = (\n    "https://videos.pexels.com/video-files/10677881/"\n    "10677881-hd_1920_1080_30fps.mp4"\n)\n\nMODEL_NAME = "yolo11s.pt"\nTRACKER_YAML = Path("/content/intersection_bytetrack.yaml")\n\nTEMP_VIDEO = Path("/content/intersection_linkedin_v2_temp.mp4")\nFINAL_VIDEO = Path("/content/intersection_linkedin_v2.mp4")\nTRACK_CSV = Path("/content/intersection_linkedin_v2_tracks.csv")\nSUMMARY_JSON = Path("/content/intersection_linkedin_v2_summary.json")\n\nOUTPUT_WIDTH = 1280\nOUTPUT_HEIGHT = 720\n\nCONFIDENCE = 0.22\nIOU = 0.55\nIMAGE_SIZE = 960\n\nINTRO_SECONDS = 3.0\nOUTRO_SECONDS = 4.0\nZONE_INTRO_SECONDS = 3.5\n\nBASE_WIDTH = 1920\nBASE_HEIGHT = 1080\n\nPERSON_CLASS = 0\nVEHICLE_CLASSES = {2, 3, 5, 7}\nTARGET_CLASSES = [0, 1, 2, 3, 5, 7]\n\n\n# ============================================================\n# VISUAL HELPERS\n# ============================================================\n\ndef scale_point(x: float, y: float, width: int, height: int) -> tuple[int, int]:\n    return int(x * width / BASE_WIDTH), int(y * height / BASE_HEIGHT)\n\n\ndef point_in_polygon(point: tuple[int, int], polygon: np.ndarray) -> bool:\n    return cv2.pointPolygonTest(polygon, point, False) >= 0\n\n\ndef alpha_rectangle(\n    frame: np.ndarray,\n    top_left: tuple[int, int],\n    bottom_right: tuple[int, int],\n    color: tuple[int, int, int],\n    alpha: float,\n) -> None:\n    overlay = frame.copy()\n    cv2.rectangle(overlay, top_left, bottom_right, color, -1)\n    cv2.addWeighted(overlay, alpha, frame, 1.0 - alpha, 0, frame)\n\n\ndef alpha_polygon(\n    frame: np.ndarray,\n    polygon: np.ndarray,\n    color: tuple[int, int, int],\n    alpha: float,\n    thickness: int = 3,\n) -> None:\n    overlay = frame.copy()\n    cv2.fillPoly(overlay, [polygon], color)\n    cv2.addWeighted(overlay, alpha, frame, 1.0 - alpha, 0, frame)\n    cv2.polylines(frame, [polygon], True, color, thickness, cv2.LINE_AA)\n\n\ndef put_text(\n    frame: np.ndarray,\n    text: str,\n    origin: tuple[int, int],\n    scale: float,\n    color: tuple[int, int, int] = (255, 255, 255),\n    thickness: int = 2,\n) -> None:\n    cv2.putText(\n        frame,\n        text,\n        origin,\n        cv2.FONT_HERSHEY_SIMPLEX,\n        scale,\n        color,\n        thickness,\n        cv2.LINE_AA,\n    )\n\n\ndef label_box(\n    frame: np.ndarray,\n    text: str,\n    origin: tuple[int, int],\n    scale: float = 0.55,\n    foreground: tuple[int, int, int] = (255, 255, 255),\n    background: tuple[int, int, int] = (15, 15, 15),\n) -> None:\n    x, y = origin\n    (text_width, text_height), baseline = cv2.getTextSize(\n        text,\n        cv2.FONT_HERSHEY_SIMPLEX,\n        scale,\n        2,\n    )\n    x = max(2, x)\n    y = max(text_height + 8, y)\n    cv2.rectangle(\n        frame,\n        (x - 4, y - text_height - 8),\n        (x + text_width + 6, y + baseline + 3),\n        background,\n        -1,\n    )\n    put_text(frame, text, (x, y), scale, foreground, 2)\n\n\ndef draw_corner_box(\n    frame: np.ndarray,\n    box: tuple[int, int, int, int],\n    color: tuple[int, int, int],\n    thickness: int = 2,\n) -> None:\n    x1, y1, x2, y2 = box\n    length = max(8, int(min(x2 - x1, y2 - y1) * 0.22))\n\n    cv2.line(frame, (x1, y1), (x1 + length, y1), color, thickness)\n    cv2.line(frame, (x1, y1), (x1, y1 + length), color, thickness)\n\n    cv2.line(frame, (x2, y1), (x2 - length, y1), color, thickness)\n    cv2.line(frame, (x2, y1), (x2, y1 + length), color, thickness)\n\n    cv2.line(frame, (x1, y2), (x1 + length, y2), color, thickness)\n    cv2.line(frame, (x1, y2), (x1, y2 - length), color, thickness)\n\n    cv2.line(frame, (x2, y2), (x2 - length, y2), color, thickness)\n    cv2.line(frame, (x2, y2), (x2, y2 - length), color, thickness)\n\n\n# ============================================================\n# TRACK STITCHING\n# ============================================================\n\nclass TrackStitcher:\n    """Reconnect short raw-tracker fragments and assign compact display IDs."""\n\n    def __init__(self, maximum_gap: int = 20) -> None:\n        self.maximum_gap = maximum_gap\n        self.next_id = 1\n        self.raw_to_canonical: dict[int, int] = {}\n        self.tracks: dict[int, dict] = {}\n\n    @staticmethod\n    def group_for_class(class_id: int) -> str:\n        if class_id == PERSON_CLASS:\n            return "person"\n        if class_id in VEHICLE_CLASSES:\n            return "vehicle"\n        return "cyclist"\n\n    def _new_track(\n        self,\n        raw_id: int,\n        class_id: int,\n        center: tuple[int, int],\n        box: tuple[int, int, int, int],\n        frame_index: int,\n    ) -> int:\n        canonical_id = self.next_id\n        self.next_id += 1\n        box_height = max(1, box[3] - box[1])\n\n        self.tracks[canonical_id] = {\n            "canonical_id": canonical_id,\n            "group": self.group_for_class(class_id),\n            "class_votes": Counter({class_id: 1}),\n            "center": center,\n            "previous_center": None,\n            "box": box,\n            "box_height": box_height,\n            "last_frame": frame_index,\n            "first_frame": frame_index,\n            "seen_frames": 1,\n            "history": deque([center], maxlen=45),\n            "counted_flow": False,\n            "counted_pedestrian": False,\n            "crosswalk_active": False,\n            "crosswalk_frames": 0,\n            "crosswalk_entry_x": None,\n            "crosswalk_min_x": center[0],\n            "crosswalk_max_x": center[0],\n            "slow_streak": 0,\n        }\n        self.raw_to_canonical[raw_id] = canonical_id\n        return canonical_id\n\n    def assign(\n        self,\n        raw_id: int,\n        class_id: int,\n        center: tuple[int, int],\n        box: tuple[int, int, int, int],\n        frame_index: int,\n        used_ids: set[int],\n    ) -> int:\n        group = self.group_for_class(class_id)\n        box_height = max(1, box[3] - box[1])\n\n        mapped_id = self.raw_to_canonical.get(raw_id)\n        canonical_id = -1\n\n        if mapped_id is not None and mapped_id in self.tracks:\n            track = self.tracks[mapped_id]\n            gap = frame_index - track["last_frame"]\n            distance = math.dist(center, track["center"])\n            allowed = max(85.0, 3.2 * max(box_height, track["box_height"]))\n\n            if (\n                mapped_id not in used_ids\n                and track["group"] == group\n                and gap <= self.maximum_gap * 3\n                and distance <= allowed\n            ):\n                canonical_id = mapped_id\n\n        if canonical_id == -1:\n            best_id = None\n            best_score = float("inf")\n\n            for candidate_id, candidate in self.tracks.items():\n                if candidate_id in used_ids or candidate["group"] != group:\n                    continue\n\n                gap = frame_index - candidate["last_frame"]\n                if gap < 1 or gap > self.maximum_gap:\n                    continue\n\n                distance = math.dist(center, candidate["center"])\n                allowed = max(\n                    55.0,\n                    2.4 * max(box_height, candidate["box_height"]),\n                )\n\n                if distance > allowed:\n                    continue\n\n                score = distance / allowed + gap * 0.025\n                if score < best_score:\n                    best_score = score\n                    best_id = candidate_id\n\n            if best_id is None:\n                canonical_id = self._new_track(\n                    raw_id,\n                    class_id,\n                    center,\n                    box,\n                    frame_index,\n                )\n                used_ids.add(canonical_id)\n                return canonical_id\n\n            canonical_id = best_id\n            self.raw_to_canonical[raw_id] = canonical_id\n\n        track = self.tracks[canonical_id]\n        track["previous_center"] = track["center"]\n        track["center"] = center\n        track["box"] = box\n        track["box_height"] = box_height\n        track["last_frame"] = frame_index\n        track["seen_frames"] += 1\n        track["class_votes"][class_id] += 1\n        track["history"].append(center)\n\n        used_ids.add(canonical_id)\n        return canonical_id\n\n    def is_stable(self, canonical_id: int, minimum_frames: int = 6) -> bool:\n        return self.tracks[canonical_id]["seen_frames"] >= minimum_frames\n\n    def movement_ratio(self, canonical_id: int, lookback: int = 12) -> float:\n        track = self.tracks[canonical_id]\n        history = track["history"]\n\n        if len(history) < lookback:\n            return float("inf")\n\n        distance = math.dist(history[-1], history[-lookback])\n        return distance / max(1.0, float(track["box_height"]))\n\n\n# ============================================================\n# SCENE CONFIGURATION\n# ============================================================\n\ndef create_scene(width: int, height: int) -> dict:\n    return {\n        "left_queue": np.array(\n            [\n                scale_point(300, 715, width, height),\n                scale_point(850, 650, width, height),\n                scale_point(930, 950, width, height),\n                scale_point(210, 950, width, height),\n            ],\n            dtype=np.int32,\n        ),\n        "right_queue": np.array(\n            [\n                scale_point(1020, 650, width, height),\n                scale_point(1510, 700, width, height),\n                scale_point(1730, 950, width, height),\n                scale_point(960, 950, width, height),\n            ],\n            dtype=np.int32,\n        ),\n        "crosswalk": np.array(\n            [\n                scale_point(0, 930, width, height),\n                scale_point(1920, 930, width, height),\n                scale_point(1920, 1080, width, height),\n                scale_point(0, 1080, width, height),\n            ],\n            dtype=np.int32,\n        ),\n        "vehicle_line_y": scale_point(0, 885, width, height)[1],\n        "median_x": scale_point(960, 0, width, height)[0],\n        "dashboard": (\n            scale_point(1440, 30, width, height),\n            scale_point(1890, 375, width, height),\n        ),\n    }\n\n\ndef relevant_detection(\n    class_id: int,\n    box: tuple[int, int, int, int],\n    frame_height: int,\n) -> bool:\n    x1, y1, x2, y2 = box\n    box_width = x2 - x1\n    box_height = y2 - y1\n    center_y = (y1 + y2) / 2\n\n    if class_id == PERSON_CLASS:\n        return (\n            center_y >= frame_height * 0.50\n            and box_height >= frame_height * 0.025\n            and box_width >= 7\n        )\n\n    return (\n        center_y >= frame_height * 0.35\n        and box_height >= frame_height * 0.018\n        and box_width >= 12\n    )\n\n\n# ============================================================\n# TITLE AND DASHBOARD\n# ============================================================\n\ndef draw_intro(frame: np.ndarray) -> np.ndarray:\n    canvas = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT))\n    canvas = cv2.GaussianBlur(canvas, (0, 0), 4.0)\n    alpha_rectangle(canvas, (0, 0), (OUTPUT_WIDTH, OUTPUT_HEIGHT), (0, 0, 0), 0.62)\n\n    put_text(canvas, "FROM CCTV TO SAFETY INSIGHT", (70, 245), 1.35, (255, 255, 255), 3)\n    cv2.line(canvas, (70, 275), (365, 275), (0, 210, 255), 6)\n\n    put_text(\n        canvas,\n        "Traffic flow  |  stopped queues  |  pedestrian activity",\n        (72, 330),\n        0.72,\n        (230, 230, 230),\n        2,\n    )\n    put_text(\n        canvas,\n        "Visual conflict candidates using YOLO and multi-object tracking",\n        (72, 370),\n        0.72,\n        (230, 230, 230),\n        2,\n    )\n    put_text(canvas, "Mohammad Amin Amiri", (72, 625), 0.70, (255, 255, 255), 2)\n    return canvas\n\n\ndef draw_dashboard(\n    frame: np.ndarray,\n    scene: dict,\n    inbound: int,\n    outbound: int,\n    left_queue: int,\n    right_queue: int,\n    pedestrian_crossings: int,\n    conflict_events: int,\n    status: str,\n) -> None:\n    (x1, y1), (x2, y2) = scene["dashboard"]\n\n    alpha_rectangle(frame, (x1, y1), (x2, y2), (8, 12, 16), 0.82)\n    cv2.rectangle(frame, (x1, y1), (x2, y2), (210, 210, 210), 2)\n\n    put_text(frame, "SMART CROSSWALK MONITOR", (x1 + 18, y1 + 38), 0.68)\n    cv2.line(frame, (x1 + 18, y1 + 55), (x2 - 18, y1 + 55), (90, 175, 220), 2)\n\n    lines = [\n        f"Traffic flow       IN {inbound:02d}  |  OUT {outbound:02d}",\n        f"Stopped queue      L {left_queue:02d}  |  R {right_queue:02d}",\n        f"Pedestrian crosses {pedestrian_crossings:02d}",\n        f"Visual alerts      {conflict_events:02d}",\n    ]\n\n    y = y1 + 94\n    for line in lines:\n        put_text(frame, line, (x1 + 18, y), 0.60, (245, 245, 245), 2)\n        y += 46\n\n    status_colors = {\n        "NORMAL": (85, 210, 125),\n        "QUEUE": (0, 205, 255),\n        "CAUTION": (0, 170, 255),\n        "ALERT": (45, 45, 235),\n    }\n    status_color = status_colors.get(status, (255, 255, 255))\n\n    alpha_rectangle(frame, (x1 + 18, y2 - 65), (x2 - 18, y2 - 18), status_color, 0.70)\n    put_text(frame, f"STATUS: {status}", (x1 + 34, y2 - 32), 0.66, (255, 255, 255), 2)\n\n\ndef draw_outro(background: np.ndarray, summary: dict) -> np.ndarray:\n    canvas = cv2.resize(background, (OUTPUT_WIDTH, OUTPUT_HEIGHT))\n    canvas = cv2.GaussianBlur(canvas, (0, 0), 5.0)\n    alpha_rectangle(canvas, (0, 0), (OUTPUT_WIDTH, OUTPUT_HEIGHT), (0, 0, 0), 0.72)\n\n    put_text(canvas, "INTERSECTION SAFETY SUMMARY", (70, 105), 1.12, (255, 255, 255), 3)\n    cv2.line(canvas, (70, 130), (360, 130), (0, 210, 255), 6)\n\n    metrics = [\n        ("Inbound vehicles", summary["inbound_vehicles"]),\n        ("Outbound vehicles", summary["outbound_vehicles"]),\n        ("Peak stopped queue", summary["peak_stopped_queue"]),\n        ("Pedestrian crossings", summary["pedestrian_crossings"]),\n        ("Visual conflict candidates", summary["visual_conflict_events"]),\n    ]\n\n    y = 200\n    for label, value in metrics:\n        put_text(canvas, label, (95, y), 0.72, (225, 225, 225), 2)\n        put_text(canvas, str(value), (560, y), 0.95, (255, 255, 255), 3)\n        y += 72\n\n    put_text(\n        canvas,\n        "A standard camera can become a practical traffic-safety sensor.",\n        (70, 590),\n        0.72,\n        (255, 255, 255),\n        2,\n    )\n    put_text(\n        canvas,\n        "Research demo: metric distances and TTC require camera calibration.",\n        (70, 635),\n        0.58,\n        (195, 195, 195),\n        1,\n    )\n    return canvas\n\n\n# ============================================================\n# MAIN PROCESSING\n# ============================================================\n\ndef main() -> None:\n    if not SOURCE_VIDEO.exists():\n        print("Downloading the intersection video...")\n        subprocess.run(\n            ["wget", "-q", "--show-progress", "-O", str(SOURCE_VIDEO), SOURCE_URL],\n            check=True,\n        )\n\n    TRACKER_YAML.write_text(\n        """tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.10\nnew_track_thresh: 0.35\ntrack_buffer: 90\nmatch_thresh: 0.85\nfuse_score: True\n""",\n        encoding="utf-8",\n    )\n\n    capture = cv2.VideoCapture(str(SOURCE_VIDEO))\n    if not capture.isOpened():\n        raise RuntimeError(f"Could not open {SOURCE_VIDEO}")\n\n    fps = float(capture.get(cv2.CAP_PROP_FPS))\n    if fps <= 0:\n        fps = 30.0\n\n    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n\n    success, first_frame = capture.read()\n    if not success:\n        capture.release()\n        raise RuntimeError("Could not read the first video frame.")\n\n    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)\n\n    writer = cv2.VideoWriter(\n        str(TEMP_VIDEO),\n        cv2.VideoWriter_fourcc(*"mp4v"),\n        fps,\n        (OUTPUT_WIDTH, OUTPUT_HEIGHT),\n    )\n    if not writer.isOpened():\n        capture.release()\n        raise RuntimeError("Could not create the temporary output video.")\n\n    intro_frame = draw_intro(first_frame)\n    for _ in range(int(round(INTRO_SECONDS * fps))):\n        writer.write(intro_frame)\n\n    device = 0 if torch.cuda.is_available() else "cpu"\n    use_half = bool(torch.cuda.is_available())\n\n    print("Device:", device)\n    if torch.cuda.is_available():\n        print("GPU:", torch.cuda.get_device_name(0))\n\n    model = YOLO(MODEL_NAME)\n    stitcher = TrackStitcher(maximum_gap=max(12, int(round(fps * 0.65))))\n    scene = create_scene(frame_width, frame_height)\n\n    inbound_count = 0\n    outbound_count = 0\n    pedestrian_crossings = 0\n    conflict_events = 0\n    peak_stopped_queue = 0\n\n    pair_streaks: dict[tuple[int, int], int] = defaultdict(int)\n    counted_pairs: set[tuple[int, int]] = set()\n    active_alert_until = -1\n    active_alert_pair: tuple[int, int] | None = None\n    last_annotated = first_frame.copy()\n\n    csv_file = TRACK_CSV.open("w", newline="", encoding="utf-8")\n    csv_writer = csv.DictWriter(\n        csv_file,\n        fieldnames=[\n            "frame", "time_seconds", "canonical_id", "raw_id", "group",\n            "confidence", "x1", "y1", "x2", "y2", "center_x", "center_y",\n            "in_left_queue", "in_right_queue", "in_crosswalk", "slow_or_stopped",\n        ],\n    )\n    csv_writer.writeheader()\n\n    frame_index = 0\n\n    while True:\n        success, frame = capture.read()\n        if not success:\n            break\n\n        result = model.track(\n            frame,\n            persist=True,\n            tracker=str(TRACKER_YAML),\n            classes=TARGET_CLASSES,\n            conf=CONFIDENCE,\n            iou=IOU,\n            imgsz=IMAGE_SIZE,\n            device=device,\n            half=use_half,\n            verbose=False,\n        )[0]\n\n        used_canonical_ids: set[int] = set()\n        visible_objects: list[dict] = []\n\n        if result.boxes is not None and result.boxes.id is not None:\n            boxes = result.boxes.xyxy.cpu().numpy()\n            class_ids = result.boxes.cls.int().cpu().tolist()\n            confidences = result.boxes.conf.cpu().tolist()\n            raw_ids = result.boxes.id.int().cpu().tolist()\n\n            for box_array, class_id, confidence, raw_id in zip(\n                boxes, class_ids, confidences, raw_ids\n            ):\n                box = tuple(map(int, box_array.tolist()))\n                if not relevant_detection(class_id, box, frame_height):\n                    continue\n\n                x1, y1, x2, y2 = box\n                center = ((x1 + x2) // 2, (y1 + y2) // 2)\n\n                canonical_id = stitcher.assign(\n                    raw_id, class_id, center, box, frame_index, used_canonical_ids\n                )\n                track = stitcher.tracks[canonical_id]\n                group = track["group"]\n                stable = stitcher.is_stable(canonical_id)\n\n                in_left = point_in_polygon(center, scene["left_queue"])\n                in_right = point_in_polygon(center, scene["right_queue"])\n                in_crosswalk = point_in_polygon(center, scene["crosswalk"])\n\n                movement_ratio = stitcher.movement_ratio(canonical_id, lookback=12)\n                slow_or_stopped = movement_ratio < 0.38\n\n                if group == "vehicle" and (in_left or in_right):\n                    if slow_or_stopped:\n                        track["slow_streak"] += 1\n                    else:\n                        track["slow_streak"] = max(0, track["slow_streak"] - 2)\n                else:\n                    track["slow_streak"] = 0\n\n                queued = stable and track["slow_streak"] >= 7\n\n                if stable and track["previous_center"] is not None:\n                    previous_x, previous_y = track["previous_center"]\n                    current_x, current_y = center\n                    line_y = scene["vehicle_line_y"]\n                    median_x = scene["median_x"]\n\n                    if group == "vehicle" and not track["counted_flow"]:\n                        crossed_inbound = current_x >= median_x and previous_y < line_y <= current_y\n                        crossed_outbound = current_x < median_x and previous_y > line_y >= current_y\n\n                        if crossed_inbound:\n                            inbound_count += 1\n                            track["counted_flow"] = True\n                        elif crossed_outbound:\n                            outbound_count += 1\n                            track["counted_flow"] = True\n\n                if group == "person":\n                    if in_crosswalk:\n                        if not track["crosswalk_active"]:\n                            track["crosswalk_active"] = True\n                            track["crosswalk_frames"] = 0\n                            track["crosswalk_entry_x"] = center[0]\n                            track["crosswalk_min_x"] = center[0]\n                            track["crosswalk_max_x"] = center[0]\n\n                        track["crosswalk_frames"] += 1\n                        track["crosswalk_min_x"] = min(track["crosswalk_min_x"], center[0])\n                        track["crosswalk_max_x"] = max(track["crosswalk_max_x"], center[0])\n\n                        horizontal_span = track["crosswalk_max_x"] - track["crosswalk_min_x"]\n                        if (\n                            stable\n                            and not track["counted_pedestrian"]\n                            and track["crosswalk_frames"] >= max(8, int(fps * 0.35))\n                            and horizontal_span >= frame_width * 0.075\n                        ):\n                            pedestrian_crossings += 1\n                            track["counted_pedestrian"] = True\n                    elif track["crosswalk_active"]:\n                        horizontal_span = track["crosswalk_max_x"] - track["crosswalk_min_x"]\n                        if (\n                            stable\n                            and not track["counted_pedestrian"]\n                            and track["crosswalk_frames"] >= max(7, int(fps * 0.30))\n                            and horizontal_span >= frame_width * 0.06\n                        ):\n                            pedestrian_crossings += 1\n                            track["counted_pedestrian"] = True\n\n                        track["crosswalk_active"] = False\n                        track["crosswalk_frames"] = 0\n\n                visible_objects.append(\n                    {\n                        "canonical_id": canonical_id,\n                        "raw_id": raw_id,\n                        "group": group,\n                        "confidence": float(confidence),\n                        "box": box,\n                        "center": center,\n                        "stable": stable,\n                        "in_left": in_left,\n                        "in_right": in_right,\n                        "in_crosswalk": in_crosswalk,\n                        "queued": queued,\n                        "movement_ratio": movement_ratio,\n                    }\n                )\n\n                csv_writer.writerow(\n                    {\n                        "frame": frame_index,\n                        "time_seconds": round(frame_index / fps, 3),\n                        "canonical_id": canonical_id,\n                        "raw_id": raw_id,\n                        "group": group,\n                        "confidence": round(float(confidence), 4),\n                        "x1": x1, "y1": y1, "x2": x2, "y2": y2,\n                        "center_x": center[0], "center_y": center[1],\n                        "in_left_queue": int(in_left),\n                        "in_right_queue": int(in_right),\n                        "in_crosswalk": int(in_crosswalk),\n                        "slow_or_stopped": int(queued),\n                    }\n                )\n\n        stable_people = [\n            obj for obj in visible_objects\n            if obj["stable"] and obj["group"] == "person" and obj["in_crosswalk"]\n        ]\n        stable_vehicles = [\n            obj for obj in visible_objects\n            if obj["stable"] and obj["group"] == "vehicle" and obj["in_crosswalk"]\n        ]\n\n        candidate_pairs: set[tuple[int, int]] = set()\n        caution_distance = frame_width * 0.12\n        high_distance = frame_width * 0.072\n\n        for person in stable_people:\n            for vehicle in stable_vehicles:\n                person_id = person["canonical_id"]\n                vehicle_id = vehicle["canonical_id"]\n                pair = (person_id, vehicle_id)\n                distance = math.dist(person["center"], vehicle["center"])\n\n                vehicle_is_moving = vehicle["movement_ratio"] > 0.16\n                closing = False\n\n                person_history = stitcher.tracks[person_id]["history"]\n                vehicle_history = stitcher.tracks[vehicle_id]["history"]\n\n                if len(person_history) >= 5 and len(vehicle_history) >= 5:\n                    previous_distance = math.dist(person_history[-5], vehicle_history[-5])\n                    closing = distance < previous_distance - 3.0\n\n                is_candidate = (\n                    distance < caution_distance\n                    and vehicle_is_moving\n                    and (closing or distance < high_distance)\n                )\n\n                if is_candidate:\n                    candidate_pairs.add(pair)\n                    pair_streaks[pair] += 1\n\n                    if pair_streaks[pair] >= max(5, int(fps * 0.18)):\n                        active_alert_until = max(\n                            active_alert_until,\n                            frame_index + int(fps * 1.35),\n                        )\n                        active_alert_pair = pair\n\n                        if pair not in counted_pairs:\n                            counted_pairs.add(pair)\n                            conflict_events += 1\n                else:\n                    pair_streaks[pair] = max(0, pair_streaks[pair] - 1)\n\n        for pair in list(pair_streaks):\n            if pair not in candidate_pairs:\n                pair_streaks[pair] = max(0, pair_streaks[pair] - 1)\n\n        left_queue_count = sum(\n            obj["queued"] and obj["in_left"]\n            for obj in visible_objects\n            if obj["group"] == "vehicle"\n        )\n        right_queue_count = sum(\n            obj["queued"] and obj["in_right"]\n            for obj in visible_objects\n            if obj["group"] == "vehicle"\n        )\n        stopped_queue_total = left_queue_count + right_queue_count\n        peak_stopped_queue = max(peak_stopped_queue, stopped_queue_total)\n\n        analysis_time = frame_index / fps\n        annotated = frame.copy()\n\n        if analysis_time < ZONE_INTRO_SECONDS:\n            fade = 1.0 - analysis_time / ZONE_INTRO_SECONDS\n            alpha_polygon(annotated, scene["left_queue"], (220, 115, 35), 0.08 + 0.14 * fade)\n            alpha_polygon(annotated, scene["right_queue"], (80, 185, 100), 0.08 + 0.14 * fade)\n            alpha_polygon(annotated, scene["crosswalk"], (45, 55, 205), 0.06 + 0.11 * fade)\n\n            label_box(\n                annotated, "STOPPED QUEUE: OUTBOUND",\n                scale_point(300, 735, frame_width, frame_height), 0.56\n            )\n            label_box(\n                annotated, "STOPPED QUEUE: INBOUND",\n                scale_point(1115, 735, frame_width, frame_height), 0.56\n            )\n            label_box(\n                annotated, "CROSSWALK MONITORING ZONE",\n                scale_point(665, 1015, frame_width, frame_height), 0.56\n            )\n        else:\n            cv2.polylines(\n                annotated, [scene["crosswalk"]], True,\n                (60, 75, 205), 2, cv2.LINE_AA\n            )\n\n        line_y = scene["vehicle_line_y"]\n        cv2.line(\n            annotated,\n            (scale_point(300, 0, frame_width, frame_height)[0], line_y),\n            (scale_point(1690, 0, frame_width, frame_height)[0], line_y),\n            (0, 205, 255), 2, cv2.LINE_AA\n        )\n\n        alert_active = frame_index <= active_alert_until\n        alert_ids = set(active_alert_pair or ()) if alert_active else set()\n\n        for obj in visible_objects:\n            if not obj["stable"]:\n                continue\n\n            canonical_id = obj["canonical_id"]\n            box = obj["box"]\n            group = obj["group"]\n            x1, y1, x2, y2 = box\n            box_height = y2 - y1\n\n            if canonical_id in alert_ids:\n                color, thickness = (35, 35, 235), 4\n            elif obj["queued"]:\n                color, thickness = (0, 180, 255), 3\n            elif group == "person":\n                color, thickness = (245, 245, 245), 2\n            elif group == "vehicle":\n                color, thickness = (80, 220, 220), 2\n            else:\n                color, thickness = (210, 120, 230), 2\n\n            draw_corner_box(annotated, box, color, thickness)\n\n            if group == "person":\n                short_label = f"P{canonical_id}"\n            elif group == "vehicle":\n                short_label = f"V{canonical_id}"\n            else:\n                short_label = f"C{canonical_id}"\n\n            if box_height >= frame_height * 0.055 or canonical_id in alert_ids:\n                label_box(\n                    annotated, short_label,\n                    (x1, max(28, y1 - 4)),\n                    0.48,\n                    background=(12, 12, 12),\n                )\n\n            if canonical_id in alert_ids:\n                history = stitcher.tracks[canonical_id]["history"]\n                points = np.array(history, dtype=np.int32).reshape((-1, 1, 2))\n                if len(points) >= 2:\n                    cv2.polylines(annotated, [points], False, color, 4, cv2.LINE_AA)\n\n        if alert_active:\n            status = "ALERT"\n        elif stopped_queue_total >= 5:\n            status = "QUEUE"\n        elif stopped_queue_total >= 2:\n            status = "CAUTION"\n        else:\n            status = "NORMAL"\n\n        draw_dashboard(\n            annotated, scene,\n            inbound_count, outbound_count,\n            left_queue_count, right_queue_count,\n            pedestrian_crossings, conflict_events, status\n        )\n\n        label_box(\n            annotated,\n            "LIVE  |  CAMERA-BASED TRAFFIC ANALYTICS",\n            scale_point(35, 55, frame_width, frame_height),\n            0.58,\n            background=(18, 23, 28),\n        )\n\n        if alert_active:\n            label_box(\n                annotated,\n                "VISUAL CONFLICT CANDIDATE",\n                scale_point(665, 95, frame_width, frame_height),\n                0.80,\n                background=(35, 35, 210),\n            )\n\n        put_text(\n            annotated,\n            "Visual analytics demo - not calibrated TTC",\n            scale_point(35, 1050, frame_width, frame_height),\n            0.48,\n            (225, 225, 225),\n            1,\n        )\n\n        writer.write(cv2.resize(annotated, (OUTPUT_WIDTH, OUTPUT_HEIGHT)))\n        last_annotated = annotated\n        frame_index += 1\n\n        if frame_index % 100 == 0:\n            print(f"{frame_index}/{total_frames} frames processed")\n\n    capture.release()\n    csv_file.close()\n\n    summary = {\n        "source_video": str(SOURCE_VIDEO),\n        "frames_processed": frame_index,\n        "video_seconds": round(frame_index / fps, 2),\n        "inbound_vehicles": inbound_count,\n        "outbound_vehicles": outbound_count,\n        "peak_stopped_queue": peak_stopped_queue,\n        "pedestrian_crossings": pedestrian_crossings,\n        "visual_conflict_events": conflict_events,\n        "note": (\n            "Visual conflict candidates are image-space screening events. "\n            "Metric TTC requires camera calibration."\n        ),\n    }\n\n    outro_frame = draw_outro(last_annotated, summary)\n    for _ in range(int(round(OUTRO_SECONDS * fps))):\n        writer.write(outro_frame)\n\n    writer.release()\n    SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")\n\n    subprocess.run(\n        [\n            "ffmpeg", "-y", "-i", str(TEMP_VIDEO),\n            "-c:v", "libx264",\n            "-preset", "veryfast",\n            "-crf", "22",\n            "-pix_fmt", "yuv420p",\n            "-movflags", "+faststart",\n            "-an", str(FINAL_VIDEO),\n        ],\n        check=True,\n        stdout=subprocess.DEVNULL,\n        stderr=subprocess.STDOUT,\n    )\n\n    print("\\nDONE")\n    print("Final video:", FINAL_VIDEO)\n    print("Track CSV:", TRACK_CSV)\n    print("Summary:", SUMMARY_JSON)\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/intersection_linkedin_v2.py').write_text(program, encoding='utf-8')
print('Program created.')

Program created.


In [7]:
!python /content/intersection_linkedin_v2.py

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/content/intersecti 100%[===================>]  10.08M  --.-KB/s    in 0.08s   
Device: 0
GPU: Tesla T4
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 256ms
Prepared 1 package in 50ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be re

In [8]:
import json
from pathlib import Path

summary_path = Path('/content/intersection_linkedin_v2_summary.json')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))

{
  "source_video": "/content/intersection.mp4",
  "frames_processed": 894,
  "video_seconds": 29.8,
  "inbound_vehicles": 51,
  "outbound_vehicles": 19,
  "peak_stopped_queue": 10,
  "pedestrian_crossings": 0,
  "visual_conflict_events": 0,
  "note": "Visual conflict candidates are image-space screening events. Metric TTC requires camera calibration."
}


In [9]:
from google.colab import files

files.download('/content/intersection_linkedin_v2.mp4')
files.download('/content/intersection_linkedin_v2_tracks.csv')
files.download('/content/intersection_linkedin_v2_summary.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>